# LeetCode 1539 — Kth Missing Positive Number bằng Python

**Link đề:** https://leetcode.com/problems/kth-missing-positive-number/
**Độ khó:** Easy · **Chủ đề:** Array, Binary Search

```python
class Solution:
    def findKthPositive(self, arr: list[int], k: int) -> int:
        ...
```

### Mục tiêu học
1. Hiểu "số nguyên dương bị thiếu thứ `k`" qua ví dụ đếm tay.
2. Dùng `set` để tra cứu nhanh và vòng `while` đếm dần.
3. Tìm ra **công thức vàng**: số lượng số bị thiếu trước vị trí `i` là `arr[i] - (i + 1)`.
4. Từ công thức đó đi tới **Binary Search** `O(log n)` — trả lời phần *Follow up* của đề.
5. Trace, chứng minh, phân tích Time/Space, test đối chiếu 4 cách, bài tập biến thể.

**Cách học:** đọc markdown → đoán output → chạy → so sánh. Cell TODO không tự chạy.

## 1. Đề bài — số dương bị thiếu thứ k

Cho mảng `arr` gồm các số nguyên dương **tăng nghiêm ngặt** và số nguyên `k`. Trả về số nguyên dương thứ `k` **không có mặt** trong `arr`.

| Ví dụ | `arr` | `k` | Các số bị thiếu | Output |
|---|---|---|---|---|
| 1 | `[2,3,4,7,11]` | 5 | `1, 5, 6, 8, 9, 10, 12, ...` | `9` |
| 2 | `[1,2,3,4]` | 2 | `5, 6, 7, ...` | `6` |

**Constraints**
- `1 <= arr.length <= 1000`, `1 <= arr[i] <= 1000`, `1 <= k <= 1000`
- `arr[i] < arr[j]` với `i < j` (tăng nghiêm ngặt, không trùng).

**Follow up:** giải nhanh hơn `O(n)`?

Câu hỏi gợi mở:
- Ví dụ 2: không có số nào thiếu **bên trong** mảng — đáp án nằm **sau** phần tử cuối. Làm sao tính mà không cần liệt kê?
- Nếu mảng đầy đủ `1..n` thì phần tử ở vị trí `i` (0-based) phải là bao nhiêu?

In [1]:
arr = [2, 3, 4, 7, 11]
present = set(arr)
missing = [x for x in range(1, 16) if x not in present]
print("Các số dương bị thiếu (đến 15):", missing)
print("Thứ 5 (k=5) là:", missing[5 - 1])

Các số dương bị thiếu (đến 15): [1, 5, 6, 8, 9, 10, 12, 13, 14, 15]
Thứ 5 (k=5) là: 9


`missing[5 - 1]` vì list đánh chỉ số từ 0: phần tử **thứ k** nằm ở chỉ số `k - 1`. Lỗi lệch-một (off-by-one) này sẽ còn gặp lại!

## 2. Nền tảng Python cần dùng

### 2.1. `set` và phép `in`

`set(arr)` tạo tập hợp; `x in s` kiểm tra **trung bình O(1)** nhờ băm. Với list, `x in lst` phải dò từng phần tử → `O(n)`.

In [2]:
arr = [2, 3, 4, 7, 11]
s = set(arr)
print(s)
print(5 in s, 7 in s)
print("list cũng được nhưng chậm hơn:", 5 in arr)

{2, 3, 4, 7, 11}
False True
list cũng được nhưng chậm hơn: False


### 2.2. Vòng `while` với điều kiện dừng

`while cond:` lặp **khi** `cond` còn đúng. Ta sẽ dùng nó để "đi" qua các số 1, 2, 3, ... cho đến khi gặp đủ `k` số bị thiếu.

In [3]:
x = 0
found = 0
while found < 3:
    x += 1
    if x % 2 == 0:          # giả sử "bị thiếu" = số chẵn
        found += 1
        print("Tìm thấy số chẵn thứ", found, "là", x)

Tìm thấy số chẵn thứ 1 là 2
Tìm thấy số chẵn thứ 2 là 4
Tìm thấy số chẵn thứ 3 là 6


### 2.3. Chỉ số 0-based và "đáng lẽ phải là"

Nếu không thiếu số nào, `arr` sẽ là `[1, 2, 3, ...]` và `arr[i] == i + 1`. Khi có số bị thiếu, `arr[i]` **lớn hơn** `i + 1`; phần chênh là số lượng số đã bị bỏ qua.

In [4]:
arr = [2, 3, 4, 7, 11]
for i, v in enumerate(arr):
    print(f"i={i}: arr[i]={v:>2}, đáng lẽ là {i+1}, chênh = {v - (i + 1)}")

i=0: arr[i]= 2, đáng lẽ là 1, chênh = 1
i=1: arr[i]= 3, đáng lẽ là 2, chênh = 1
i=2: arr[i]= 4, đáng lẽ là 3, chênh = 1
i=3: arr[i]= 7, đáng lẽ là 4, chênh = 3
i=4: arr[i]=11, đáng lẽ là 5, chênh = 6


### 2.4. `//` — chia lấy phần nguyên

Binary search cần điểm giữa `mid = (lo + hi) // 2`. `//` bỏ phần thập phân (làm tròn xuống).

In [5]:
print(7 // 2, 8 // 2, (0 + 4) // 2, (3 + 4) // 2)

3 4 2 3


## 3. Cách 1 — Đếm dần với `set` (trực giác)

Đi qua các số `1, 2, 3, ...`. Số nào **không** có trong `set(arr)` thì đếm. Khi đếm đủ `k`, trả về số đó.

In [6]:
def kth_missing_set(arr: list[int], k: int) -> int:
    present = set(arr)
    x = 0
    missing_count = 0
    while missing_count < k:
        x += 1
        if x not in present:
            missing_count += 1
    return x

print(kth_missing_set([2, 3, 4, 7, 11], 5))   # 9
print(kth_missing_set([1, 2, 3, 4], 2))       # 6

9
6


**Chi phí:** biến `x` tăng tới đáp án, mà đáp án ≤ `n + k` (trong `n + k` số đầu tiên có tối đa `n` số thuộc `arr`, nên ít nhất `k` số bị thiếu). Vậy Time `O(n + k)`, Space `O(n)` cho `set`.

Nút thắt: ta "đi bộ" qua **từng số**, kể cả những đoạn dài không có gì đặc biệt (ví dụ `arr = [1000]`, `k = 1000` phải đi gần 1001 bước).

## 4. Cách 2 — Duyệt mảng, "đẩy" k lên

Quan sát: mỗi phần tử `num` của `arr` mà `num <= k` (hiện tại) sẽ "chiếm chỗ" một số dương trong khoảng cần xét → đáp án bị đẩy lên 1.

```text
ans = k
for num in arr:
    if num <= ans: ans += 1     # num chiếm chỗ, đẩy đáp án lên
    else: break                 # mảng tăng, các num sau càng lớn
```

In [7]:
def kth_missing_shift(arr: list[int], k: int) -> int:
    ans = k
    for num in arr:
        if num <= ans:
            ans += 1
        else:
            break
    return ans

print(kth_missing_shift([2, 3, 4, 7, 11], 5))
print(kth_missing_shift([1, 2, 3, 4], 2))

9
6


Trace ví dụ 1 (`k = 5`):

| num | `ans` trước | `num <= ans`? | `ans` sau |
|---|---|---|---|
| 2 | 5 | ✓ | 6 |
| 3 | 6 | ✓ | 7 |
| 4 | 7 | ✓ | 8 |
| 7 | 8 | ✓ | 9 |
| 11 | 9 | ✗ → dừng | 9 |

Kết quả `9` ✓. Time `O(n)`, Space `O(1)`.

## 5. Bước đột phá — công thức số bị thiếu

Định nghĩa: `missing(i) = arr[i] - (i + 1)` = **số số dương bị thiếu nhỏ hơn `arr[i]`**.

Vì sao? Trong `1..arr[i]` có `arr[i]` số; trong đó `i + 1` số có mặt (`arr[0..i]`, đều ≤ `arr[i]` vì mảng tăng). Phần còn lại bị thiếu.

Tính chất quan trọng: `missing(i)` **không giảm** theo `i` (vì `arr[i+1] >= arr[i] + 1` → `missing(i+1) >= missing(i)`). Mảng "không giảm" → **binary search được**!

In [8]:
arr = [2, 3, 4, 7, 11]
miss = [arr[i] - (i + 1) for i in range(len(arr))]
print("arr    :", arr)
print("missing:", miss, "← không giảm")

arr    : [2, 3, 4, 7, 11]
missing: [1, 1, 1, 3, 6] ← không giảm


Đọc bảng: trước `11` có 6 số bị thiếu (1,5,6,8,9,10). Với `k = 5`: `missing(3) = 3 < 5 <= missing(4) = 6` → đáp án nằm **giữa** `arr[3] = 7` và `arr[4] = 11`.

**Công thức đáp án:** gọi `p` là số phần tử có `missing < k` (chính là vị trí đầu tiên có `missing >= k`). Khi đó trước đáp án có đúng `p` phần tử của `arr` đứng chen vào, nên:

```text
answer = p + k
```

Kiểm tra ví dụ 1: `p = 4` (các vị trí 0..3 có missing 1,1,1,3 < 5) → `4 + 5 = 9` ✓. Ví dụ 2: missing toàn 0 < 2 → `p = 4` → `4 + 2 = 6` ✓.

### 5.1. Cách 3 — Dùng công thức + duyệt tuyến tính

In [9]:
def kth_missing_linear_formula(arr: list[int], k: int) -> int:
    p = 0
    while p < len(arr) and arr[p] - (p + 1) < k:
        p += 1
    return p + k

print(kth_missing_linear_formula([2, 3, 4, 7, 11], 5))
print(kth_missing_linear_formula([1, 2, 3, 4], 2))
print(kth_missing_linear_formula([5, 6], 2))      # thiếu 1,2,3,4 → thứ 2 là 2

9
6
2


Chú ý thứ tự trong `while p < len(arr) and ...`: kiểm tra `p < len(arr)` **trước** để không truy cập `arr[len(arr)]` — nhờ `and` ngắn mạch.

## 6. Lời giải chính — Binary Search trên `missing`

### 6.1. Từng đoạn
1. Tìm vị trí `p` đầu tiên có `missing(p) >= k` (nếu không có, `p = n`). Đây là bài toán **"tìm biên trái"** (lower bound).
2. Khoảng tìm kiếm `[lo, hi) = [0, n)`, giữ **bất biến**: mọi chỉ số `< lo` có `missing < k`, mọi chỉ số `>= hi` có `missing >= k`.
3. Lặp khi `lo < hi`: `mid = (lo + hi) // 2`.
   - Nếu `missing(mid) < k` → đáp án ở bên phải: `lo = mid + 1`.
   - Ngược lại → `mid` có thể là đáp án: `hi = mid`.
4. Khi `lo == hi`, đó là `p`. Trả về `lo + k`.

### 6.2. Ghép lại

In [10]:
def kth_missing_binary(arr: list[int], k: int) -> int:
    lo, hi = 0, len(arr)
    while lo < hi:
        mid = (lo + hi) // 2
        if arr[mid] - (mid + 1) < k:
            lo = mid + 1
        else:
            hi = mid
    return lo + k

print(kth_missing_binary([2, 3, 4, 7, 11], 5))
print(kth_missing_binary([1, 2, 3, 4], 2))
print(kth_missing_binary([5, 6], 2))

9
6
2


## 7. TRACE Binary Search

Ví dụ 1: `arr = [2,3,4,7,11]`, `k = 5`, `missing = [1,1,1,3,6]`.

| Bước | lo | hi | mid | missing(mid) | so với k=5 | Hành động |
|---|---|---|---|---|---|---|
| 1 | 0 | 5 | 2 | 1 | < 5 | lo = 3 |
| 2 | 3 | 5 | 4 | 6 | ≥ 5 | hi = 4 |
| 3 | 3 | 4 | 3 | 3 | < 5 | lo = 4 |
| — | 4 | 4 | | | dừng | trả `4 + 5 = 9` |

In [11]:
def trace_binary(arr, k):
    lo, hi = 0, len(arr); step = 1
    print("missing =", [arr[i] - (i + 1) for i in range(len(arr))], "| k =", k)
    while lo < hi:
        mid = (lo + hi) // 2
        m = arr[mid] - (mid + 1)
        if m < k:
            action = f"lo = {mid + 1}"
            lo = mid + 1
        else:
            action = f"hi = {mid}"
            hi = mid
        print(f"Bước {step}: mid={mid}, missing={m} {'<' if m < k else '>='} {k} → {action}  (lo={lo}, hi={hi})")
        step += 1
    print(f"Dừng: lo = {lo} → đáp án = {lo} + {k} = {lo + k}")
    return lo + k

assert trace_binary([2, 3, 4, 7, 11], 5) == 9
print()
assert trace_binary([1, 2, 3, 4], 2) == 6
print()
assert trace_binary([5, 6, 7], 3) == 3

missing = [1, 1, 1, 3, 6] | k = 5
Bước 1: mid=2, missing=1 < 5 → lo = 3  (lo=3, hi=5)
Bước 2: mid=4, missing=6 >= 5 → hi = 4  (lo=3, hi=4)
Bước 3: mid=3, missing=3 < 5 → lo = 4  (lo=4, hi=4)
Dừng: lo = 4 → đáp án = 4 + 5 = 9

missing = [0, 0, 0, 0] | k = 2
Bước 1: mid=2, missing=0 < 2 → lo = 3  (lo=3, hi=4)
Bước 2: mid=3, missing=0 < 2 → lo = 4  (lo=4, hi=4)
Dừng: lo = 4 → đáp án = 4 + 2 = 6

missing = [4, 4, 4] | k = 3
Bước 1: mid=1, missing=4 >= 3 → hi = 1  (lo=0, hi=1)
Bước 2: mid=0, missing=4 >= 3 → hi = 0  (lo=0, hi=0)
Dừng: lo = 0 → đáp án = 0 + 3 = 3


Ba trace minh họa ba tình huống:
- Đáp án **giữa** mảng (`9`).
- Đáp án **sau** phần tử cuối (`6`): `lo` chạy tới `n = 4`.
- Đáp án **trước** phần tử đầu (`3`, vì thiếu 1,2,3,4): `missing(0) = 4 >= 3` → `lo` dừng ở 0 → `0 + 3 = 3`.

## 8. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space |
|---|---|---|---|
| 1. Set + đếm | đi từng số 1,2,3,... | `O(n + k)` | `O(n)` |
| 2. Đẩy `k` | `num <= ans` → `ans += 1` | `O(n)` | `O(1)` |
| 3. Công thức + tuyến tính | tìm `p` đầu tiên `missing >= k` | `O(n)` | `O(1)` |
| 4. Binary search | như 3 nhưng chia đôi | **`O(log n)`** | `O(1)` |

## 9. Code nộp LeetCode

In [12]:
class Solution:
    def findKthPositive(self, arr: list[int], k: int) -> int:
        lo, hi = 0, len(arr)
        while lo < hi:
            mid = (lo + hi) // 2
            if arr[mid] - (mid + 1) < k:   # trước arr[mid] thiếu chưa đủ k số
                lo = mid + 1
            else:
                hi = mid
        return lo + k

sol = Solution()
print(sol.findKthPositive([2, 3, 4, 7, 11], 5))
print(sol.findKthPositive([1, 2, 3, 4], 2))

9
6


Phiên bản `O(n)` ngắn gọn (cũng Accepted):

```python
class Solution:
    def findKthPositive(self, arr: list[int], k: int) -> int:
        for num in arr:
            if num <= k:
                k += 1
            else:
                break
        return k
```

## 10. Vì sao đúng?

**Công thức `answer = p + k`:** Gọi `p` = số phần tử của `arr` có `missing < k`, tức các phần tử `arr[0..p-1]` đều đứng **trước** số bị thiếu thứ `k` (trước chúng thiếu chưa tới `k` số). Còn `arr[p..]` có `missing >= k` → đứng **sau** đáp án. Vậy trong `1..answer` có đúng `p` số có mặt và `k` số bị thiếu → `answer = p + k`.

**Binary search — invariant:** luôn có `missing(i) < k` với mọi `i < lo`, và `missing(i) >= k` với mọi `i >= hi`.
- Khởi tạo `lo = 0, hi = n`: hai tập rỗng → đúng.
- Nếu `missing(mid) < k`: vì `missing` không giảm, mọi `i <= mid` cũng `< k` → gán `lo = mid + 1` giữ invariant.
- Ngược lại `missing(mid) >= k` → mọi `i >= mid` cũng `>= k` → `hi = mid` giữ invariant.
- Mỗi vòng khoảng `[lo, hi)` co lại ít nhất 1 → dừng. Khi `lo == hi`, `lo` chính là `p`.

**Cách 2:** sau khi xử lý các `num <= ans`, `ans` luôn bằng `k + (số phần tử đã gặp ≤ ans)`; khi gặp `num > ans`, mọi phần tử sau còn lớn hơn → không ảnh hưởng.

## 11. Time / Space

Gọi `n = len(arr)`.

- **Binary search:** mỗi vòng chia đôi khoảng `[lo, hi)` → khoảng `⌈log₂(n+1)⌉` vòng → **Time `O(log n)`**, **Space `O(1)`**. Với `n = 1000` chỉ ~10 vòng.
- Cách 2/3: `O(n)` time, `O(1)` space.
- Cách 1: `O(n + k)` time, `O(n)` space.

Đếm số vòng thực tế:

In [13]:
def count_iterations(n, k):
    arr = list(range(2, 2 * n + 2, 2))   # các số chẵn: thiếu toàn số lẻ
    lo, hi, it = 0, len(arr), 0
    while lo < hi:
        it += 1
        mid = (lo + hi) // 2
        if arr[mid] - (mid + 1) < k:
            lo = mid + 1
        else:
            hi = mid
    return it

for n in [10, 100, 1000, 10**6]:
    print(f"n={n:>7}: số vòng binary search = {count_iterations(n, n // 2)}")

n=     10: số vòng binary search = 4
n=    100: số vòng binary search = 7
n=   1000: số vòng binary search = 10
n=1000000: số vòng binary search = 20


## 12. Bộ test — mẫu đề, biên, phản ví dụ, đối chiếu

In [14]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

sol = Solution()
check("đề 1", sol.findKthPositive([2, 3, 4, 7, 11], 5), 9)
check("đề 2", sol.findKthPositive([1, 2, 3, 4], 2), 6)
check("thiếu ngay từ đầu", sol.findKthPositive([5, 6, 7], 3), 3)
check("n=1, có số 1", sol.findKthPositive([1], 1), 2)
check("n=1, không có số 1", sol.findKthPositive([2], 1), 1)
check("k max, arr max", sol.findKthPositive([1000], 1000), 1001)
check("đủ 1..1000", sol.findKthPositive(list(range(1, 1001)), 1000), 2000)
check("đúng tại biên giữa", sol.findKthPositive([2, 3, 4, 7, 11], 3), 6)

PASS đề 1: 9
PASS đề 2: 6
PASS thiếu ngay từ đầu: 3
PASS n=1, có số 1: 2
PASS n=1, không có số 1: 1
PASS k max, arr max: 1001
PASS đủ 1..1000: 2000
PASS đúng tại biên giữa: 6


### 12.1. Đối chiếu 4 cách trên input ngẫu nhiên

`rng.sample(range(1, 60), m)` chọn `m` số **khác nhau**; `sorted` để mảng tăng nghiêm ngặt.

In [15]:
import random
rng = random.Random(1539)
for _ in range(1000):
    m = rng.randint(1, 30)
    arr = sorted(rng.sample(range(1, 60), m))
    k = rng.randint(1, 50)
    exp = kth_missing_set(arr, k)
    assert kth_missing_shift(arr, k) == exp
    assert kth_missing_linear_formula(arr, k) == exp
    assert kth_missing_binary(arr, k) == exp
    assert Solution().findKthPositive(arr, k) == exp
print("1000 test ngẫu nhiên: 4 cách khớp nhau")

1000 test ngẫu nhiên: 4 cách khớp nhau


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Quên `+ 1` trong công thức missing
```python
arr[mid] - mid < k        # sai: đếm như thể số thứ 0 là 0
```
Đếm lệch 1 → sai mỗi khi có `missing(i) == k - 1` (đáp án nằm ngay sau `arr[i]`). Ca cuối tình cờ đúng — vì thế cần nhiều test!

In [16]:
def kth_wrong_formula(arr, k):
    lo, hi = 0, len(arr)
    while lo < hi:
        mid = (lo + hi) // 2
        if arr[mid] - mid < k:       # SAI
            lo = mid + 1
        else:
            hi = mid
    return lo + k

for arr, k in [([2, 3, 4, 7, 11], 4), ([2], 2), ([2, 3, 4, 7, 11], 5)]:
    print(arr, k, "| sai:", kth_wrong_formula(arr, k), "| đúng:", kth_missing_binary(arr, k))

[2, 3, 4, 7, 11] 4 | sai: 7 | đúng: 8
[2] 2 | sai: 2 | đúng: 3
[2, 3, 4, 7, 11] 5 | sai: 9 | đúng: 9


### Lỗi 2 — Trả về `arr[lo] + ...` hoặc `arr[lo-1] + ...` không kiểm tra biên
Một công thức tương đương là `arr[lo-1] + (k - missing(lo-1))`, nhưng khi `lo = 0` thì `arr[-1]` là **phần tử cuối** (Python cho chỉ số âm!) → sai âm thầm. `lo + k` gọn và không cần xử lý biên.

In [17]:
arr = [5, 6, 7]
print("arr[-1] trong Python là:", arr[-1], "(phần tử cuối, không phải lỗi!)")

arr[-1] trong Python là: 7 (phần tử cuối, không phải lỗi!)


### Lỗi 3 — Dùng `hi = len(arr) - 1` với điều kiện `lo < hi`
Khi đó khoảng tìm không bao giờ chứa `n`, nên trường hợp đáp án **sau** phần tử cuối (ví dụ 2) bị sai.

In [18]:
def kth_wrong_hi(arr, k):
    lo, hi = 0, len(arr) - 1           # SAI: thiếu khả năng p = n
    while lo < hi:
        mid = (lo + hi) // 2
        if arr[mid] - (mid + 1) < k:
            lo = mid + 1
        else:
            hi = mid
    return lo + k

print("sai:", kth_wrong_hi([1, 2, 3, 4], 2), "| đúng:", kth_missing_binary([1, 2, 3, 4], 2))

sai: 5 | đúng: 6


### Lỗi 4 — Vòng lặp vô hạn khi viết `lo = mid`
Nếu nhánh trái viết `lo = mid` (thay vì `mid + 1`), khi `hi = lo + 1` thì `mid = lo` → không tiến → lặp mãi. Quy tắc: với `mid = (lo+hi)//2` (làm tròn xuống), nhánh tăng `lo` **phải** là `mid + 1`.

### Lỗi 5 — Cách 2 quên `break` / dùng `<` thay `<=`
`if num < ans` bỏ sót trường hợp `num == ans` (số đó có mặt nên phải đẩy). Ví dụ `[1]`, `k=1`: đáp án 2, nhưng `1 < 1` sai → trả 1.

In [19]:
def kth_shift_wrong(arr, k):
    ans = k
    for num in arr:
        if num < ans:        # SAI: phải là <=
            ans += 1
    return ans

print("sai:", kth_shift_wrong([1], 1), "| đúng:", kth_missing_shift([1], 1))

sai: 1 | đúng: 2


## 14. Mẹo nhận dạng pattern

- Mảng **tăng nghiêm ngặt** + câu hỏi "thiếu bao nhiêu / thứ k bị thiếu" → nghĩ ngay `arr[i] - (i + 1)` (hoặc `arr[i] - arr[0] - i` nếu bắt đầu từ `arr[0]`).
- Có một đại lượng **đơn điệu** theo chỉ số (`missing(i)` không giảm) → **binary search trên chỉ số**, kể cả khi đại lượng đó không lưu sẵn trong mảng.
- "Follow up: nhanh hơn O(n)" trên mảng đã sắp → gần như chắc chắn là binary search.
- Template **lower bound** `[lo, hi)`: `hi = n`, `while lo < hi`, `lo = mid + 1` / `hi = mid`. Thuộc lòng để khỏi lỗi lệch-một.

## 15. Biến thể

### 15.1. Đếm số bị thiếu trong đoạn `[1, x]`
Bằng `x - (số phần tử <= x)`. Số phần tử `<= x` tìm bằng `bisect_right`.

`bisect.bisect_right(a, x)` trả về vị trí chèn `x` vào bên phải các phần tử bằng `x` = số phần tử `<= x` (mảng đã sắp).

In [20]:
from bisect import bisect_right
arr = [2, 3, 4, 7, 11]
for x in [1, 5, 9, 11, 15]:
    print(f"Số bị thiếu trong [1,{x}] = {x - bisect_right(arr, x)}")

Số bị thiếu trong [1,1] = 1
Số bị thiếu trong [1,5] = 2
Số bị thiếu trong [1,9] = 5
Số bị thiếu trong [1,11] = 6
Số bị thiếu trong [1,15] = 10


### 15.2. Số bị thiếu thứ k **bắt đầu từ `arr[0]`** (LeetCode 1060, Premium)
Không đếm các số nhỏ hơn `arr[0]`. Công thức: `missing(i) = arr[i] - arr[0] - i`.

In [21]:
def kth_missing_from_first(nums, k):
    lo, hi = 0, len(nums)
    while lo < hi:
        mid = (lo + hi) // 2
        if nums[mid] - nums[0] - mid < k:
            lo = mid + 1
        else:
            hi = mid
    return nums[0] + lo - 1 + k

print(kth_missing_from_first([4, 7, 9, 10], 1))   # 5
print(kth_missing_from_first([4, 7, 9, 10], 3))   # 8
print(kth_missing_from_first([1, 2, 4], 3))       # 6

5
8
6


Kiểm tra: `[4,7,9,10]`, các số thiếu sau 4: `5, 6, 8, 11, ...` → thứ 1 là 5, thứ 3 là 8 ✓. Đáp án `nums[0] + lo - 1 + k`: giống `p + k` nhưng "dịch" gốc từ 1 sang `nums[0]`.

### 15.3. Số dương nhỏ nhất bị thiếu khi mảng **chưa sắp** (LeetCode 41)
Đó là bài Hard "First Missing Positive" — dùng hoán vị tại chỗ, không dùng binary search được vì mất tính đơn điệu.

## 16. Bài tập (TODO không tự chạy)

### Bài 1 — Tính tay
`arr = [3, 5, 6, 10]`. Tính `missing(i)` cho mọi `i` và tìm số thiếu thứ `k = 4`.

### Bài 2 — Đoán output
```python
arr = [1, 3]
print([arr[i] - (i + 1) for i in range(len(arr))])
```

### Bài 3 — TODO: cách "đẩy k"

In [22]:
def exercise_shift(arr, k):
    # TODO: duyệt num trong arr; num <= k thì k += 1, ngược lại dừng
    pass

# assert exercise_shift([2, 3, 4, 7, 11], 5) == 9
# assert exercise_shift([1, 2, 3, 4], 2) == 6

### Bài 4 — TODO: binary search

In [23]:
def exercise_binary(arr, k):
    # TODO 1: lo, hi = 0, len(arr)
    # TODO 2: while lo < hi: so sánh arr[mid] - (mid + 1) với k
    # TODO 3: return lo + k
    pass

# assert exercise_binary([2, 3, 4, 7, 11], 5) == 9
# assert exercise_binary([5, 6, 7], 3) == 3

### Bài 5 — TODO: liệt kê `k` số bị thiếu đầu tiên
Trả về list `k` số dương bị thiếu nhỏ nhất, thời gian `O(n + k)`, **không dùng set**.

In [24]:
def exercise_first_k_missing(arr, k):
    # TODO: hai con trỏ: x chạy 1,2,3,...; i chỉ vào arr
    pass

# assert exercise_first_k_missing([2, 3, 4, 7, 11], 5) == [1, 5, 6, 8, 9]
# assert exercise_first_k_missing([1, 2, 3, 4], 2) == [5, 6]

### Bài 6 — Mini challenge: nhiều truy vấn
Cho `arr` và một danh sách `queries` gồm nhiều giá trị `k`. Trả về đáp án cho từng `k`, mỗi truy vấn `O(log n)`. Thử với `bisect` thay cho vòng `while` tự viết.

In [25]:
def exercise_many_queries(arr, queries):
    # TODO: tính trước mảng missing (không giảm), dùng bisect_left(missing, k)
    pass

# assert exercise_many_queries([2, 3, 4, 7, 11], [1, 2, 5, 6, 7]) == [1, 5, 9, 10, 12]

## 17. Gợi ý 3 mức

**Bài 1:** (1) `missing(i) = arr[i] - (i+1)`. (2) Tìm `p` đầu tiên có `missing >= 4`. (3) `answer = p + 4`.

**Bài 2:** (1) Áp công thức. (2) `1 - 1`, `3 - 2`. (3) Một list hai phần tử.

**Bài 3:** (1) Mỗi số có mặt ≤ đáp án đẩy đáp án lên. (2) `for ... if ... else: break`. (3) Xem mục 4.

**Bài 4:** (1) Lower bound trên `missing`. (2) `lo = mid + 1` khi `< k`, ngược lại `hi = mid`. (3) Xem mục 6.

**Bài 5:** (1) Đi `x = 1, 2, ...`, con trỏ `i` trên `arr`. (2) Nếu `i < n and arr[i] == x` thì `i += 1` (có mặt), ngược lại thêm `x`. (3) Dừng khi đủ `k`.

**Bài 6:** (1) `missing` không giảm → dùng được `bisect_left`. (2) `bisect_left(missing, k)` = vị trí đầu tiên `missing >= k` = `p`. (3) Mỗi truy vấn trả `p + k`.

## 18. Đáp án tham khảo

**Bài 1:** `missing = [2, 3, 3, 6]` (thiếu 1,2 trước 3; thêm 4 trước 5; thêm 7,8,9 trước 10). Vị trí đầu tiên `missing >= 4` là `p = 3` → đáp án `3 + 4 = 7`. Kiểm tra: các số thiếu `1, 2, 4, 7, ...` → thứ 4 là 7 ✓.

**Bài 2:** `[0, 1]`.

In [26]:
from bisect import bisect_left

def answer_shift(arr, k):
    for num in arr:
        if num <= k:
            k += 1
        else:
            break
    return k

def answer_binary(arr, k):
    lo, hi = 0, len(arr)
    while lo < hi:
        mid = (lo + hi) // 2
        if arr[mid] - (mid + 1) < k:
            lo = mid + 1
        else:
            hi = mid
    return lo + k

def answer_first_k_missing(arr, k):
    res = []; i = 0; x = 0
    while len(res) < k:
        x += 1
        if i < len(arr) and arr[i] == x:
            i += 1
        else:
            res.append(x)
    return res

def answer_many_queries(arr, queries):
    missing = [arr[i] - (i + 1) for i in range(len(arr))]
    return [bisect_left(missing, k) + k for k in queries]

assert answer_shift([2, 3, 4, 7, 11], 5) == 9
assert answer_shift([1, 2, 3, 4], 2) == 6
assert answer_binary([2, 3, 4, 7, 11], 5) == 9
assert answer_binary([5, 6, 7], 3) == 3
assert answer_binary([3, 5, 6, 10], 4) == 7
assert answer_first_k_missing([2, 3, 4, 7, 11], 5) == [1, 5, 6, 8, 9]
assert answer_first_k_missing([1, 2, 3, 4], 2) == [5, 6]
assert answer_many_queries([2, 3, 4, 7, 11], [1, 2, 5, 6, 7]) == [1, 5, 9, 10, 12]
rng = random.Random(7)
for _ in range(300):
    arr = sorted(rng.sample(range(1, 80), rng.randint(1, 30)))
    qs = [rng.randint(1, 60) for _ in range(5)]
    assert answer_many_queries(arr, qs) == [kth_missing_set(arr, q) for q in qs]
    assert answer_first_k_missing(arr, qs[0])[-1] == kth_missing_set(arr, qs[0])
print("Tất cả đáp án PASS")

Tất cả đáp án PASS


## 19. Template nhớ nhanh + checklist submit

```python
class Solution:
    def findKthPositive(self, arr: list[int], k: int) -> int:
        lo, hi = 0, len(arr)                  # [lo, hi), hi = n để bao cả "sau phần tử cuối"
        while lo < hi:
            mid = (lo + hi) // 2
            if arr[mid] - (mid + 1) < k:      # missing(mid) < k
                lo = mid + 1
            else:
                hi = mid
        return lo + k                         # p phần tử có mặt + k số thiếu
```

**Câu thần chú:** *"Thiếu trước arr[i] = arr[i] − (i+1); đáp án = (số phần tử đứng trước) + k."*

**Checklist trước khi submit**
- [ ] Công thức có `+ 1`: `arr[mid] - (mid + 1)`.
- [ ] `hi = len(arr)` (không phải `len(arr) - 1`).
- [ ] Nhánh trái là `lo = mid + 1` (tránh lặp vô hạn).
- [ ] Test 3 tình huống: đáp án trước phần tử đầu, giữa mảng, sau phần tử cuối.
- [ ] Đúng tên method `findKthPositive`.